In [1]:
import sys
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

sys.path.append(os.path.abspath(".."))
from src.ETL import PROCESSED_DIR, MODELS_DIR

In [2]:
# 1. Load the Clean Data directly
df = pd.read_csv(os.path.join(PROCESSED_DIR, "historical_clean_bhopal.csv"))

features = ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'hour', 'month']
target_col = 'temperature_2m'

In [3]:
# 2. Scale the data
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[features])

# Save the scaler (overwriting the old one)
scaler_path = os.path.join(MODELS_DIR, "scaler_bhopal.pkl")
joblib.dump(scaler, scaler_path)
print(f"Scaler saved to {scaler_path}")

Scaler saved to c:\Users\HP\Desktop\PROJECTS\weather-elt-project\models\scaler_bhopal.pkl


In [4]:
# 3. Create Multi-Step Sequences (120 hours)
seq_length = 48
forecast_horizon = 120
target_idx = features.index(target_col)

X, y = [], []
for i in range(len(scaled_data) - seq_length - forecast_horizon):
    X.append(scaled_data[i : i + seq_length])
    # The target is 120 hours of temperature
    y.append(scaled_data[i + seq_length : i + seq_length + forecast_horizon, target_idx])
    
X = np.array(X)
y = np.array(y)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}") # THIS MUST SAY (Samples, 120)

X shape: (17352, 48, 6)
y shape: (17352, 120)


In [5]:
# 4. Split Data
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

In [6]:
# 5. Build the Multi-Step Model
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X.shape[1], X.shape[2])),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(120) # <--- 120 outputs for 120 hours
])

model.compile(optimizer='adam', loss='mean_squared_error')

c:\Users\HP\anaconda3\envs\eltenv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [7]:
# 6. Train
print("Training Multi-Step Model...")
history = model.fit(
    X_train, y_train, 
    epochs=10, 
    batch_size=32, 
    validation_data=(X_test, y_test)
)

Training Multi-Step Model...
Epoch 1/10
434/434 ━━━━━━━━━━━━━━━━━━━━ 35s 65ms/step - loss: 0.0264 - val_loss: 0.0056
Epoch 2/10
434/434 ━━━━━━━━━━━━━━━━━━━━ 27s 49ms/step - loss: 0.0049 - val_loss: 0.0020
Epoch 3/10
434/434 ━━━━━━━━━━━━━━━━━━━━ 41s 49ms/step - loss: 0.0040 - val_loss: 0.0018
Epoch 4/10
434/434 ━━━━━━━━━━━━━━━━━━━━ 23s 52ms/step - loss: 0.0037 - val_loss: 0.0018
Epoch 5/10
434/434 ━━━━━━━━━━━━━━━━━━━━ 22s 52ms/step - loss: 0.0036 - val_loss: 0.0019
Epoch 6/10
434/434 ━━━━━━━━━━━━━━━━━━━━ 21s 49ms/step - loss: 0.0036 - val_loss: 0.0016
Epoch 7/10
434/434 ━━━━━━━━━━━━━━━━━━━━ 21s 49ms/step - loss: 0.0035 - val_loss: 0.0018
Epoch 8/10
434/434 ━━━━━━━━━━━━━━━━━━━━ 22s 50ms/step - loss: 0.0035 - val_loss: 0.0019
Epoch 9/10
434/434 ━━━━━━━━━━━━━━━━━━━━ 22s 49ms/step - loss: 0.0034 - val_loss: 0.0018
Epoch 10/10
434/434 ━━━━━━━━━━━━━━━━━━━━ 21s 49ms/step - loss: 0.0034 - val_loss: 0.0016


In [8]:
# 7. Save
model_path = os.path.join(MODELS_DIR, "weather_lstm_bhopal_multistep.keras")
model.save(model_path)
print(f"✅ Multi-Step Model saved to {model_path}")

✅ Multi-Step Model saved to c:\Users\HP\Desktop\PROJECTS\weather-elt-project\models\weather_lstm_bhopal_multistep.keras
